# Preparing flight data for AutoGluon

Load separate datasets for training, validation, calibration, and testing. 

| Dataset | Default scheduled departure period (UTC) | Later purpose |
| --- | --- | --- |
| Training | 2019–2022 | Fit the model |
| Validation | 2023 | Compare models and settings |
| Calibration | 2024 | Choose a decision threshold after model selection |
| Test | Available 2025 records | Evaluate the frozen model and threshold |

## 1. Choose the airports and periods

Set `ORIGIN_AIRPORTS` to a list of origin airport codes, for example `['DFW', 'ORD']`. A one-airport list such as `['DFW']` also works. Use `None` to load all origins. An empty list is rejected to avoid accidentally loading an unintended cohort.

Every column is loaded. More airports mean more rows and higher memory use. The default keeps DFW only; add airports as needed.

End dates are **exclusive**: a flight departing at exactly the validation start belongs to validation, not training. Keep the four periods ordered and non-overlapping.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Start in the project root or notebooks directory.'

ORIGIN_AIRPORTS = ['DFW']  # Examples: ['DFW', 'ORD']; None for all origins.
if ORIGIN_AIRPORTS is not None:
    if not isinstance(ORIGIN_AIRPORTS, list) or not ORIGIN_AIRPORTS:
        raise ValueError('ORIGIN_AIRPORTS must be a non-empty list or None.')
    if not all(isinstance(airport, str) and airport.strip() for airport in ORIGIN_AIRPORTS):
        raise ValueError('Each airport must be a non-empty string.')
    ORIGIN_AIRPORTS = list(dict.fromkeys(airport.strip().upper() for airport in ORIGIN_AIRPORTS))
PERIODS = {
    'training': ('2019-01-01', '2023-01-01'),
    'validation': ('2023-01-01', '2024-01-01'),
    'calibration': ('2024-01-01', '2025-01-01'),
    'test': ('2025-01-01', '2026-01-01'),
}

previous_end = None
for name, (start, end) in PERIODS.items():
    assert pd.Timestamp(start) < pd.Timestamp(end), f'{name}: start must precede end.'
    if previous_end is not None:
        assert pd.Timestamp(previous_end) <= pd.Timestamp(start), 'Periods must not overlap.'
    previous_end = end

DATA = ROOT / 'data' / 'bts_flights_weather' / 'flights_weather'
files = sorted(DATA.glob('year=*/*.parquet'))
assert files, 'Download the BTS flight/weather year partitions first.'
con = duckdb.connect()
con.execute("SET threads = 4")
con.execute("SET TimeZone = 'UTC'")
con.execute("SET memory_limit = '2GB'")
con.read_parquet([str(file) for file in files], hive_partitioning=False).create_view('flights')
print('Origin airports:', ORIGIN_AIRPORTS if ORIGIN_AIRPORTS is not None else 'all origins')
print('Available year folders:', sorted({file.parent.name for file in files}))

Origin airports: ['DFW']
Available year folders: ['year=2019', 'year=2020', 'year=2021', 'year=2022', 'year=2023', 'year=2024', 'year=2025']


## 2. Load all columns

Use `SELECT *` to retain every column supplied in the Parquet data, including scheduled information, weather measurements, timestamps, and flight outcomes. This stage only loads data; feature selection and the target definition will come later.

Loading a column does not make it a suitable model input. Before training, separate the target and exclude outcomes, `oracle_` columns, and any information unavailable at the prediction time.

The airport filter keeps departures from any airport in `ORIGIN_AIRPORTS`. `None` disables the airport filter. Query parameters pass dates and airport codes separately from the SQL text.

The result is a pandas DataFrame with Arrow-backed columns to reduce memory use when loading all columns.

In [2]:
LOAD_COLUMNS = con.execute("DESCRIBE flights").fetchdf()['column_name'].tolist()
print('Columns to load:', len(LOAD_COLUMNS))

def load_period(start, end):
    """Load all columns for one UTC interval and the selected origin airports."""
    result = con.sql("""
        SELECT *
        FROM flights
        WHERE sched_dep_utc >= CAST(? AS TIMESTAMPTZ)
          AND sched_dep_utc < CAST(? AS TIMESTAMPTZ)
          AND (? IS NULL OR list_contains(?, origin_iata))
        ORDER BY sched_dep_utc
    """, params=[start, end, ORIGIN_AIRPORTS, ORIGIN_AIRPORTS])
    return result.to_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)

Columns to load: 210


## 3. Load the four datasets

These names will be used in the later AutoGluon steps. No model is fitted and no preprocessing is learned from validation, calibration, or test data.

In [3]:
train_data = load_period(*PERIODS['training'])
validation_data = load_period(*PERIODS['validation'])
calibration_data = load_period(*PERIODS['calibration'])
test_data = load_period(*PERIODS['test'])

In [4]:
train_data.head(5)

,operating_carrier,flight_number,tail_number,origin_iata,dest_iata,dep_delay_min,arr_delay_min,taxi_out_min,taxi_in_min,crs_elapsed_min,...,oracle_dest_arr_obs_time_utc,oracle_origin_dep_obs_minus_event_min,oracle_origin_dep_is_post_event,oracle_origin_dep_is_post_cutoff,oracle_dest_scharr_obs_minus_event_min,oracle_dest_scharr_is_post_event,oracle_dest_scharr_is_post_cutoff,oracle_dest_arr_obs_minus_event_min,oracle_dest_arr_is_post_event,oracle_dest_arr_is_post_cutoff
0,AA,2812,N846NN,DFW,ORD,-3.0,-35.0,10.0,10.0,145.0,...,2019-01-01 13:51:00+00:00,46,True,True,16,True,True,51,True,True
1,AA,745,N917UY,DFW,CLT,-5.0,0.0,32.0,8.0,146.0,...,2019-01-01 13:52:00+00:00,43,True,True,11,True,True,11,True,True
2,AA,1042,N906AA,DFW,MIA,-1.0,-6.0,12.0,10.0,162.0,...,2019-01-01 14:53:00+00:00,29,True,True,46,True,True,52,True,True
3,F9,129,N927FR,DFW,DEN,-5.0,-27.0,14.0,10.0,139.0,...,2019-01-01 13:53:00+00:00,13,True,True,49,True,True,16,True,True
4,OO,5994,N203SY,DFW,SFO,-3.0,-14.0,21.0,22.0,255.0,...,2019-01-01 16:56:00+00:00,-4,False,False,41,True,True,55,True,True
